# Task 1: Iris Flower Classification

**Track:** Data Science
**Objective:** Train a machine learning classification model to identify the species of an iris flower (Setosa, Versicolor, or Virginica) from its physical measurements.

**Tech Stack:** Python, scikit-learn, pandas, matplotlib/seaborn, Jupyter Notebook

## 1. Load the Iris Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

# Load dataset
df = pd.read_csv('iris.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
display(df.head())
print(f'\nColumn Names: {list(df.columns)}')
print(f'\nData Types:')
print(df.dtypes)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== DATASET INFO ===')
print(df.info())

# Null value check
print('\n=== NULL VALUES ===')
print(df.isnull().sum())

# Descriptive statistics
print('\n=== DESCRIPTIVE STATISTICS ===')
display(df.describe())

# Species distribution
print('\n=== SPECIES DISTRIBUTION ===')
print(df['species'].value_counts())

# Class balance check
print(f'\nClass balance: {df["species"].value_counts().min()} samples per class (balanced)')

## 3. Visualizations

In [ ]:
# Pairplot - feature distributions by species
sns.pairplot(df, hue='species', palette='husl', diag_kind='hist')
plt.suptitle('Iris Features Pairplot by Species', y=1.02, fontweight='bold')
plt.show()

In [ ]:
# Box plots for each feature by species
features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.boxplot(data=df, x='species', y=feat, ax=axes[i], palette='husl')
    axes[i].set_title(f'{feat} by Species', fontweight='bold')
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel(feat)

plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for better distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.violinplot(data=df, x='species', y=feat, ax=axes[i], palette='husl', inner='box')
    axes[i].set_title(f'{feat} Distribution by Species', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Feature Selection Discussion

### Which features are most discriminative?

From the visualizations:

1. **Petal length (cm)** and **Petal width (cm)** - Show clear separation between all three species with minimal overlap. Setosa is distinctly separate, Versicolor and Virginica have some overlap but are distinguishable.

2. **Sepal length (cm)** - Moderate separation. Setosa is distinct, but Versicolor and Virginica overlap significantly.

3. **Sepal width (cm)** - Least discriminative. Significant overlap between all three species, though Setosa tends to have wider sepals.

**Conclusion:** Petal measurements are the most discriminative features for classifying iris species. Sepal width is the least useful.

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
numeric_df = df[features + ['target']]
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

print('=== FEATURE CORRELATIONS ===')
print(corr[features].round(2))

## 5. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare features and target
X = df[features]
y = df['target']  # Using numeric target for modeling

# Train/test split (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size: {X_test.shape[0]}')
print(f'\nTrain class distribution:')
print(pd.Series(y_train).value_counts().sort_index())
print(f'\nTest class distribution:')
print(pd.Series(y_test).value_counts().sort_index())

## 6. Train Multiple Classifiers

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': acc
    }
    print(f'{name}: Accuracy = {acc:.4f}')

## 7. Detailed Model Evaluation

In [ ]:
species_names = ['setosa', 'versicolor', 'virginica']

for name, result in results.items():
    print(f'\n=== {name.upper()} ===')
    print(f'Accuracy: {result["accuracy"]:.4f}')
    
    # Classification report
    print('\nClassification Report:')
    print(classification_report(y_test, result['predictions'], target_names=species_names))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, result['predictions'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=species_names, yticklabels=species_names)
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

## 8. Model Comparison & Best Model Selection

In [ ]:
# Compare all models
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[name]['accuracy'] for name in results.keys()]
}).sort_values('Accuracy', ascending=False)

print('=== MODEL COMPARISON ===')
display(comparison)

# Visualize comparison
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x='Accuracy', y='Model', palette='viridis')
plt.title('Model Accuracy Comparison', fontweight='bold')
plt.xlim(0.9, 1.0)
for i, v in enumerate(comparison['Accuracy']):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Best model
best_model_name = comparison.iloc[0]['Model']
best_accuracy = comparison.iloc[0]['Accuracy']
print(f'\n🏆 BEST MODEL: {best_model_name} with Accuracy = {best_accuracy:.4f}')

## 9. Feature Importance (for tree-based models)

In [ ]:
# Feature importance for Random Forest and Decision Tree
for name in ['Random Forest', 'Decision Tree']:
    if name in results:
        model = results[name]['model']
        importances = model.feature_importances_
        
        imp_df = pd.DataFrame({
            'Feature': features,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        print(f'\n=== {name.upper()} FEATURE IMPORTANCE ===')
        display(imp_df)
        
        plt.figure(figsize=(8, 5))
        sns.barplot(data=imp_df, x='Importance', y='Feature', palette='viridis')
        plt.title(f'{name} - Feature Importance', fontweight='bold')
        plt.tight_layout()
        plt.show()

## 10. Conclusion

### Summary

1. **Dataset**: 150 samples, 4 features, 3 classes (perfectly balanced - 50 each)

2. **EDA Findings**:
   - Petal length and petal width are the most discriminative features
   - Setosa is linearly separable from the other two species
   - Versicolor and Virginica have some overlap in sepal measurements
   - Strong correlation between petal length and petal width (0.96)

3. **Model Performance**:
   - All models achieve >95% accuracy on this well-separated dataset
   - Random Forest and Logistic Regression typically perform best
   - Even simple models like KNN work well due to clear class separation

4. **Best Model**: **[Best Model Name]** with **[Accuracy]%** accuracy

5. **Key Insight**: The Iris dataset is a classic "hello world" for classification - the classes are well-separated, making it ideal for learning but not representative of real-world noisy data.